In [1]:
import re
import pandas as pd

In [2]:
# ── 1. Load ───────────────────────────────────────────────────────────────────
flows   = pd.read_csv("flow_logs_data.csv")
advisor = pd.read_csv("advisor_recommendations.csv")

In [3]:
# ── 2. Normalise flow VM columns → short key ──────────────────────────────────
def vm_to_key(val):
    """'rg-jxd-ddd/flowdemo-web2'  →  'flowdemo-web2'
       raw IP or empty              →  None  (won't match)
    """
    if pd.isna(val) or not str(val).strip():
        return None
    return str(val).split("/")[-1].strip().lower()

flows["src_key"] = flows["SrcVm"].apply(vm_to_key)
flows["dst_key"] = flows["DestVm"].apply(vm_to_key)

In [4]:
# ── 3. Normalise ImpactedValue → vm key ───────────────────────────────────────
_FLOWDEMO_RE = re.compile(r"(flowdemo-[a-z0-9]+)", re.IGNORECASE)

def impacted_to_key(val):
    """Extract the leading 'flowdemo-<name>' token, lowercase.

    Examples
    --------
    'flowdemo-app1'                                    → 'flowdemo-app1'
    'FLOWDEMO-WEB2'                                    → 'flowdemo-web2'
    'flowdemo-app1_osdisk_1_39fa323c...'               → 'flowdemo-app1'
    'flowdemo-app1-nic'                                → 'flowdemo-app1'
    'flowdemoflowlogs4xnyiget'                         → None  (storage acct)
    '3ca6ed87-dfc5-...'                                → None  (subscription)
    """
    if pd.isna(val):
        return None
    m = _FLOWDEMO_RE.search(str(val))
    return m.group(1).lower() if m else None

advisor["vm_key"] = advisor["ImpactedValue"].apply(impacted_to_key)

# Preview the key mapping
print("=== Advisor vm_key sample ===")
print(advisor[["ImpactedValue", "vm_key"]].drop_duplicates().to_string(index=False))

=== Advisor vm_key sample ===
                                           ImpactedValue               vm_key
 flowdemo-app4_osdisk_1_884d9d8f897945da9d8d224043b6adbf        flowdemo-app4
 flowdemo-app1_osdisk_1_39fa323cf3a940c8b631f46cbdc866ec        flowdemo-app1
 flowdemo-app2_osdisk_1_524125070d704f9db010e015faac341f        flowdemo-app2
 flowdemo-app3_osdisk_1_72e885cb18a042ff9a0c28f7df0034cf        flowdemo-app3
                    3ca6ed87-dfc5-4dce-9cf7-71c84695304d                  NaN
                                flowdemoflowlogs4xnyiget                  NaN
                                           flowdemo-app2        flowdemo-app2
                                           FLOWDEMO-WEB2        flowdemo-web2
                                           flowdemo-app1        flowdemo-app1
                                    FLOWDEMO-PGSECONDARY flowdemo-pgsecondary
                                          flowdemo-applb       flowdemo-applb
                                  

In [5]:
# ── 4. Explode flows: join on SrcVm AND DestVm separately, then combine ───────
#  Strategy: for each flow row we want recommendations that match *either*
#  the source or the destination VM.  We do two left-joins and union them.

advisor_keyed = advisor.dropna(subset=["vm_key"])  # skip non-VM advisor rows for join


In [6]:
# 4a. Match on destination VM (most flows are inbound with only DestVm populated)
dst_join = flows.merge(
    advisor_keyed.add_prefix("adv_"),
    left_on="dst_key",
    right_on="adv_vm_key",
    how="left",
)
dst_join["_match_side"] = "DestVm"

# 4b. Match on source VM
src_join = flows.merge(
    advisor_keyed.add_prefix("adv_"),
    left_on="src_key",
    right_on="adv_vm_key",
    how="left",
)
src_join["_match_side"] = "SrcVm"

# 4c. Combine and de-duplicate identical (flow_row × advisor_row) pairs
combined = (
    pd.concat([dst_join, src_join], ignore_index=True)
    .drop_duplicates(
        subset=[c for c in dst_join.columns if c != "_match_side"]
    )
    .reset_index(drop=True)
)

In [7]:
# ── 5. Summary ────────────────────────────────────────────────────────────────
matched   = combined.dropna(subset=["adv_ImpactedValue"])
unmatched = combined[combined["adv_ImpactedValue"].isna()]

print(f"\nTotal joined rows  : {len(combined):,}")
print(f"  with advisor match : {len(matched):,}")
print(f"  without match      : {len(unmatched):,}")

print("\n=== Advisor recommendations matched per VM ===")
print(
    matched
    .groupby(["adv_vm_key", "adv_Category", "adv_Impact"])["adv_Description"]
    .first()
    .reset_index()
    .sort_values(["adv_vm_key", "adv_Impact"])
    .to_string(index=False)
)




Total joined rows  : 6,044
  with advisor match : 4,642
  without match      : 1,402

=== Advisor recommendations matched per VM ===
          adv_vm_key          adv_Category adv_Impact                                                                                    adv_Description
       flowdemo-app1      HighAvailability       High          Ensure Azure Disks are in the same zone as your VM for higher resiliency and availability
       flowdemo-app1 OperationalExcellence       High Enable Trusted Launch foundational excellence, and modern security for Existing Generation 2 VM(s)
       flowdemo-app1      HighAvailability     Medium                                                Migrate workload to Virtual Machine Scale Sets Flex
       flowdemo-app1 OperationalExcellence     Medium                                           Add explicit outbound method to disable default outbound
       flowdemo-app2      HighAvailability       High          Ensure Azure Disks are in the same zon

In [12]:
# ── 6. Add total traffic per advisor VM key ───────────────────────────────────
combined["_row_traffic"] = (
    combined["BytesSrcToDest"].fillna(0) + combined["BytesDestToSrc"].fillna(0)
)

combined["total_traffic"] = combined.groupby("adv_vm_key")["_row_traffic"].transform("sum")
combined = combined.drop(columns=["_row_traffic"])

print("\n=== High impact recommendations with total traffic per adv_vm_key ===")
print(
    combined[[
        "adv_vm_key",
        "adv_Category",
        "adv_Impact",
        "adv_Description",
        "total_traffic",
    ]]
    .dropna(subset=["adv_vm_key"])
    .query("adv_Impact == 'High'")
    .drop_duplicates()
    .sort_values(
        ["total_traffic", "adv_vm_key", "adv_Category"],
        ascending=[False, True, True],
    )
    .to_string(index=False)
)


=== High impact recommendations with total traffic per adv_vm_key ===
          adv_vm_key          adv_Category adv_Impact                                                                                    adv_Description  total_traffic
       flowdemo-web2 OperationalExcellence       High Enable Trusted Launch foundational excellence, and modern security for Existing Generation 2 VM(s)     15508569.0
       flowdemo-app1      HighAvailability       High          Ensure Azure Disks are in the same zone as your VM for higher resiliency and availability      1606140.0
       flowdemo-app1 OperationalExcellence       High Enable Trusted Launch foundational excellence, and modern security for Existing Generation 2 VM(s)      1606140.0
  flowdemo-pgprimary OperationalExcellence       High Enable Trusted Launch foundational excellence, and modern security for Existing Generation 2 VM(s)      1453476.0
      flowdemo-applb OperationalExcellence       High Enable Trusted Launch foundational 

In [13]:
# ── 6. Export ─────────────────────────────────────────────────────────────────
combined.to_csv("high_pri_advisor.csv", index=False)
print("\nFull joined table saved to: high_pri_advisor.csv")


Full joined table saved to: high_pri_advisor.csv
